# 00 — Setup

**Purpose.** This notebook holds every import, file path, helper function, and
feature-engineering routine that the other notebooks reuse. Nothing trains here —
this is the toolbox.

**How to use it.** In every other notebook the first cell is `%run 00_setup.ipynb`.
That literally re-executes the cells of this file in the same Python process, so
all the variables and functions defined here become available in the caller.
It's the cheap, no-import-path version of a Python package — fine for prototyping,
ugly for production. When we ship to the KFP pipeline (Phase 4) we copy these
functions into `src/features/build.py` and `src/train/train.py`.

**Why a separate "setup" notebook at all?**
Notebook cells share global state. Re-running everything from scratch becomes
slow. By isolating the slow imports + dataset loading into one place, we can
iterate on features (notebook 02) or models (notebook 03) without rerunning
boilerplate every time.


## Step 1 — Install Python packages

The Jupyter pod runs the PyTorch CUDA image, which ships with `torch`,
`numpy`, basic stdlib — but **not** pandas, xgboost, shap, or matplotlib.
We install with `--user` so the files land in `/workspace/.local`, which is
a hostPath mount on the K8s node: packages survive pod restarts and reinstalls
are cached on hostPath too, so this is fast on second runs.

`!` in front of a line tells Jupyter "run this in a shell, not Python".
`2>&1 | tail -3` swallows the 100-line pip output and keeps only the last
three lines (the "Successfully installed..." summary).


In [1]:
# Persistent user install — survives pod restarts via hostPath /workspace/.local
!pip install --user --no-cache-dir pandas numpy pyarrow xgboost shap matplotlib seaborn 2>&1 | tail -3

## Step 2 — Imports and file paths

- `pandas` (`pd`) — table library; everything is a `DataFrame` (rows × cols).
- `numpy` (`np`) — math on arrays (`np.log`, `np.sin`, etc.).
- `matplotlib.pyplot` (`plt`) — plotting. The `figsize` default of 12×4 inches
  gives a wide chart that fits a notebook cell well.
- `pathlib.Path` — modern path handling; works on Linux/macOS/Windows.

**About the data layout.** The CSV files live in `notebooks/data/`. When this
file runs from inside `notebooks/`, `Path.cwd()` is the notebooks directory,
so `HERE / "data"` resolves correctly. The `if (HERE / "data").exists()` check
is a safety net so the same code works if you ever move the notebooks one
level up.

**Stale-flag and ETF columns.**
The dataset-service injects `*_age_hours` columns next to every alt-data field
("how stale is this number, in hours?"). In a live system that's useful — if
hashrate is 48 hours old, you trust it less. In our **backfill** the snapshot
service always reports `0` because every row was reconstructed from history at
once, so these columns are constant zero → no signal → drop them.
`etf_flow_total` is empty before the Jan 2024 ETF launch (~70% of rows are
NaN). Until the ETF series is longer, keeping it adds noise; we drop it by
default. Both can be re-enabled via the function flags below.


In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Wide-and-shallow charts read nicely in Jupyter cells.
plt.rcParams["figure.figsize"] = (12, 4)

# Where the CSV snapshots live. `Path.cwd()` = directory Jupyter was launched from.
HERE = Path.cwd()
DATA_DIR = HERE / "data" if (HERE / "data").exists() else HERE

CSV_PATH    = DATA_DIR / "BTCUSDT_1d_merged.csv"       # 2019-05 → 2025-12 (train+val+test)
HOLDOUT_CSV = DATA_DIR / "BTCUSDT_1d_2026_holdout.csv" # 2026, untouched until final eval

# Columns we drop by default — see explanation above.
ETF_COLS = ["etf_flow_total"]
AGE_COLS = [
    "hash_rate_age_hours","difficulty_age_hours","block_time_age_hours",
    "miner_reserves_age_hours","miner_outflows_age_hours","mpi_age_hours",
    "exchange_inflow_age_hours","exchange_outflow_age_hours",
    "exchange_reserves_age_hours","etf_flow_total_age_hours",
]

## Step 3 — `load_dataset()`

Read the CSV from disk, parse the timestamp column as UTC datetimes, sort
chronologically, drop the dead columns. Sorting is defensive — we expect the
CSV to be in order, but never trust input data.

**Why UTC?** Crypto markets don't close. There's no "session timezone".
Everything in this project uses UTC end-to-end (CSV, BigQuery, model serving)
to avoid timezone bugs.

**Parameters:**
- `drop_age_cols=True` → drop the stale-flag columns (constant 0 in backfill).
- `drop_etf=True` → drop the mostly-empty ETF column.

Override either flag to keep the column.


In [3]:
def load_dataset(path=CSV_PATH, drop_age_cols=True, drop_etf=True):
    df = pd.read_csv(path)

    # Parse "2019-05-01T00:00:00Z" → pandas datetime, UTC.
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    # Belt-and-braces: ensure chronological order. ML on time-series MUST stay sorted.
    df = df.sort_values("timestamp").reset_index(drop=True)

    if drop_age_cols:
        df = df.drop(columns=[c for c in AGE_COLS if c in df.columns])
    if drop_etf:
        df = df.drop(columns=[c for c in ETF_COLS if c in df.columns])
    elif "etf_flow_total" in df.columns:
        # Pre-2024 = no ETFs existed = no flow. Filling 0 is semantically correct.
        df["etf_flow_total"] = df["etf_flow_total"].fillna(0.0)
    return df


## Step 4 — `drop_indicator_warmup()`

Technical indicators are **rolling windows**: RSI(14) needs 14 past closes,
EMA200 needs 200, Ichimoku's `senkou_b` needs 52 weeks (~365 days). For the
first ~365 rows of the dataset, these slow indicators are either zero or
mathematically meaningless.

If we trained on those rows the model would learn "when RSI = 0, market does
X" — pure noise. Standard practice on real desks: cut the head of the
backfill until every rolling feature is warmed up. One year is the safe
choice for this feature set.


In [4]:
def drop_indicator_warmup(df, warmup_days=365):
    cutoff = df["timestamp"].min() + pd.Timedelta(days=warmup_days)
    return df[df["timestamp"] >= cutoff].reset_index(drop=True)


## Step 5 — `add_derived_features()`

The CSV ships with ~30 technical indicators (RSI, MACD, Bollinger, …). Those
are **descriptive** — they tell you what already happened. We add a few
**transformed** features that often help an ML model find signal:

### Log returns (momentum)
`log(close / close.shift(k))` = the *k*-day log return. We compute 1, 3, 5,
10, 20. Three things to know about log returns:

1. They're additive over time — 3 daily log returns sum to the 3-day log return.
2. They're roughly symmetric around 0, unlike percent returns (-50% to recover
   needs +100%).
3. ML models prefer scale-invariant inputs. `(close=87000, prev_close=86000)`
   carries no relative meaning to a tree; `log(87000/86000) = +0.0116` does.

### Realized volatility (`vol_5/20/60`)
Standard deviation of recent daily log-returns. Low = calm market, high =
turbulent. Models often learn very different rules in calm vs turbulent
regimes, so they need a way to see "what regime am I in".

### Regime features
- `atr_pct = ATR / close` — Average True Range as a percentage of price.
  Same idea as vol but uses high-low ranges instead of close-to-close.
- `bb_width` and `kelt_width` — width of Bollinger / Keltner bands relative
  to mid. Wider band = wider expected swing.
- `px_vs_sma` — how far above/below the 200-day average we are.
  Positive = uptrend, negative = downtrend.
- `sma_vs_ema` — SMA and EMA disagree in turning points; their ratio tracks
  trend acceleration.

### Volume regime
`vol_z_20` — today's volume z-scored against the last 20 days. Catches
volume spikes (whale dumps, breakouts).

### Calendar features (cyclical encoding)
Day-of-week and month are **circular** — December and January are next to
each other, but encoded as 12 and 1 they look maximally far apart. Sin/cos
encoding gives `(sin, cos)` pairs that wrap around correctly.


In [5]:
def add_derived_features(df):
    df = df.copy()

    # --- 1) Log returns over multiple horizons (momentum at different timescales)
    for k in (1, 3, 5, 10, 20):
        df[f"logret_{k}"] = np.log(df["close"] / df["close"].shift(k))

    # --- 2) Realized volatility (rolling stdev of 1-day returns)
    df["vol_5"]  = df["logret_1"].rolling(5).std()
    df["vol_20"] = df["logret_1"].rolling(20).std()
    df["vol_60"] = df["logret_1"].rolling(60).std()

    # --- 3) Price-scale-normalized regime features
    df["atr_pct"]    = df["atr"] / df["close"]
    df["bb_width"]   = (df["bollinger_upper"] - df["bollinger_lower"]) / df["bollinger_middle"]
    df["kelt_width"] = (df["keltner_upper"]   - df["keltner_lower"])   / df["keltner_middle"]

    # --- 4) Trend strength
    df["px_vs_sma"]   = df["close"] / df["sma"] - 1   # +0.05 = price 5% above SMA
    df["sma_vs_ema"]  = df["sma"]   / df["ema"] - 1   # SMA vs EMA disagreement

    # --- 5) Volume regime (today's volume z-scored against 20-day mean)
    df["vol_z_20"] = (df["volume"] - df["volume"].rolling(20).mean()) / df["volume"].rolling(20).std()

    # --- 6) Cyclical calendar encoding (so December ~= January)
    dt = df["timestamp"].dt
    df["dow_sin"]   = np.sin(2*np.pi*dt.dayofweek/7)
    df["dow_cos"]   = np.cos(2*np.pi*dt.dayofweek/7)
    df["month_sin"] = np.sin(2*np.pi*(dt.month-1)/12)
    df["month_cos"] = np.cos(2*np.pi*(dt.month-1)/12)

    return df


## Step 6 — Labels: regression vs classification

A supervised ML model learns `f(features) → label`. We have to decide what
the label is.

- **Regression** (`label_regression`): predict a *number*. Here, the H-day
  forward **log return**: `log(close[t+H] / close[t])`. From that we can
  reconstruct the price: `predicted_price = close[t] * exp(prediction)`.
  This is what we want for "predict price in N days".

- **Classification** (`label_classification`): predict a *category*. Here:
  did the price go up by more than `EPS` (e.g. 0.5%)? We keep this function
  around for sanity-checking direction-only setups, but the main task is
  regression.

The `.shift(-H)` look-ahead is the **only** place in the pipeline where we
look forward in time. After labeling, the bottom `H` rows have NaN labels
(no future data), so we drop them with `dropna(subset=...)`.


In [6]:
def label_regression(df, H=7):
    """H-day forward log-return. Reconstruct price: close[t] * exp(prediction)."""
    df = df.copy()
    df["y_logret"] = np.log(df["close"].shift(-H) / df["close"])
    df["y_price"]  = df["close"].shift(-H)
    return df.dropna(subset=["y_logret"]).reset_index(drop=True)

def label_classification(df, H=1, EPS=0.005):
    """Binary up/down with dead-band. Kept for sanity comparison only."""
    df = df.copy()
    df["y"] = (df["close"].shift(-H) > df["close"] * (1 + EPS)).astype("Int64")
    return df.dropna(subset=["y"]).reset_index(drop=True)


## Step 7 — `feature_columns()`

Picks every numeric column **except**:
- `timestamp` / `symbol` / `interval` → identifiers, not features.
- Labels (`y`, `y_logret`, `y_price`) → must never be used as a feature
  (that would be looking at the answer key).

We **do** keep `open / high / low / close / volume` in the feature set even
though they're correlated with the label, because the model needs to know
the current price level to interpret indicator values. The leak is prevented
elsewhere by the time-based split (you never see future closes during
training).


In [7]:
def feature_columns(df, exclude=None):
    drop = {"timestamp","symbol","interval","y","y_logret","y_price"}
    if exclude: drop |= set(exclude)
    return [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]

# Sanity print so when you %run this notebook you can see the toolbox loaded:
print("loaders ready:",
      load_dataset.__name__,
      drop_indicator_warmup.__name__,
      add_derived_features.__name__,
      label_regression.__name__,
      label_classification.__name__,
      feature_columns.__name__)


loaders ready: load_dataset drop_indicator_warmup add_derived_features label_regression label_classification feature_columns
